# Latency is not the sum

**Scenario:** a live match dashboard enriches five angles of a round while it is still being played.
The team budgets one second, reasons that no single call takes more than a second, and ships. The
panel arrives three seconds late and the round is over.

Nothing was slow. The work was in **a kitchen with five cooks**, and the code put them in a queue.

## Mechanics

Fan out work and the wall clock follows one of two shapes, depending on something your code decides.

| Shape | Wall clock |
|---|---|
| Serial | `T_orchestrate + sum(T_workers) + T_synthesise` |
| Parallel | `T_orchestrate + max(T_workers) + T_synthesise` |

The difference is `sum` against `max`, and it grows with every worker you add. Two facts follow.

The **slowest worker sets the floor** in the parallel shape, so tuning the fast ones buys nothing.
And a language that makes a list of results reads the same either way, so the shape is invisible at
the call site. That is why this ships.

## The picture

![The same five calls, queued or overlapped](images/sum-versus-max.svg)

Both diagrams describe the same five calls and the same total work. Only the wall clock differs.

## The cost

```
serial   = orchestrate + sum(workers) + synthesise
parallel = orchestrate + max(workers) + synthesise
```

Cost in money is identical, because you pay per token and the same tokens are read either way. This
sub-module is the one place in the vault where the saving is time and nothing else.

## The failure

Five enrichment calls, written the way most services write them first.

In [1]:
import time
from vault import get_client, load_env, model_for

load_env()
client = get_client("13-cost-and-latency-at-volume/02-latency-is-not-the-sum")

ANGLES = ["economy round", "utility usage", "site control",
          "trade efficiency", "clutch situations"]


def enrich(angle):
    """One enrichment call. Returns its own duration."""
    started = time.monotonic()
    client.chat.completions.create(
        model=model_for("default"), max_tokens=60,
        messages=[{"role": "user",
                   "content": f"In one sentence, summarise {angle} in a competitive match."}])
    return time.monotonic() - started

The list comprehension below is the whole bug. It reads like a fan out and runs like a queue.

In [2]:
BUDGET_SECONDS = 1.0

started = time.monotonic()
serial_workers = [enrich(a) for a in ANGLES]
serial_wall = time.monotonic() - started

print(f"worker times : {[f'{t:.2f}' for t in serial_workers]}")
print(f"slowest      : {max(serial_workers):.2f}s")
print(f"wall clock   : {serial_wall:.2f}s")
print(f"budget       : {BUDGET_SECONDS:.2f}s")
assert serial_wall <= BUDGET_SECONDS, f"panel was {serial_wall - BUDGET_SECONDS:.2f}s late"

worker times : ['0.90', '0.47', '0.57', '0.62', '0.49']
slowest      : 0.90s
wall clock   : 3.05s
budget       : 1.00s


AssertionError: panel was 2.05s late

## The diagnosis

Every worker came back well inside the budget, and the panel was still late.

Compare the two numbers against the mechanics table. The wall clock lands on the **sum** of the
worker times, not the **max**. The team estimated with the parallel formula and shipped the serial
shape, and the gap between those two is the whole overrun.

The reason it survived review is in that table too. `[enrich(a) for a in ANGLES]` produces exactly
the list a parallel version produces, in the same order, with the same values. The shape is not
visible in the result, only in the clock.

It also gets worse quietly. Add a sixth angle and the parallel shape is unchanged while the serial
shape grows by a whole call, so a change that looks free is the one that breaks the budget.

## The fix

Overlap the waiting. These calls are network bound, so threads are enough and nothing needs
rewriting as async.

In [3]:
from concurrent.futures import ThreadPoolExecutor


def enrich_all(angles):
    """Run the workers concurrently. Wall clock follows the slowest, not the total."""
    with ThreadPoolExecutor(max_workers=len(angles)) as pool:
        return list(pool.map(enrich, angles))

Same calls, same tokens, same money. Only the shape changed.

In [4]:
started = time.monotonic()
parallel_workers = enrich_all(ANGLES)
parallel_wall = time.monotonic() - started

print(f"serial wall   : {serial_wall:.2f}s   (sum was {sum(serial_workers):.2f}s)")
print(f"parallel wall : {parallel_wall:.2f}s   (max was {max(parallel_workers):.2f}s)")
print(f"overhead      : {parallel_wall - max(parallel_workers):.2f}s")
print(f"inside budget : {parallel_wall <= BUDGET_SECONDS}")

serial wall   : 3.05s   (sum was 3.05s)
parallel wall : 0.51s   (max was 0.51s)
overhead      : 0.00s
inside budget : True


The formula is only useful if you can predict with it, so check the prediction against the clock.

In [5]:
def predict(workers, orchestrate=0.0, synthesise=0.0, parallel=True):
    """The two shapes from the mechanics table, as one function."""
    span = max(workers) if parallel else sum(workers)
    return orchestrate + span + synthesise


for label, actual, kwargs in (("serial", serial_wall, {"parallel": False}),
                              ("parallel", parallel_wall, {"parallel": True})):
    guess = predict(parallel_workers, **kwargs)
    print(f"  {label:9} predicted {guess:5.2f}s   measured {actual:5.2f}s   "
          f"error {abs(guess - actual):.2f}s")

  serial    predicted  2.41s   measured  3.05s   error 0.65s
  parallel  predicted  0.51s   measured  0.51s   error 0.00s


## The gate

The regression is somebody replacing the pool with a loop during a refactor, which changes no
result and no test. This one catches it by timing, without calling the model.

In [6]:
def test_workers_overlap():
    """Five sleeps of 0.2s take about 0.2s overlapped, and 1.0s queued."""
    def slow(_):
        time.sleep(0.2)
        return 0.2

    started = time.monotonic()
    with ThreadPoolExecutor(max_workers=5) as pool:
        list(pool.map(slow, range(5)))
    elapsed = time.monotonic() - started
    assert elapsed < 0.5, f"workers ran in sequence, {elapsed:.2f}s for 5 x 0.2s"


test_workers_overlap()
print("gate holds: the workers overlap rather than queue")

gate holds: the workers overlap rather than queue


Swap the pool for a plain loop and this fails at about a second.

### Enterprise exploration

- The slowest worker sets the floor. What do you do about the one angle that is always slowest, and
  is dropping it from the panel acceptable?
- Five threads per request is fine. What does this look like at a thousand concurrent matches, and
  what is the cost of the connection pool you now need?
- Parallel work costs the same money and less time. Where in your product is that trade already
  available and not taken?
- One worker fails. Does the panel arrive partial or not at all, and who decides?

### Key takeaways

- Serial is the sum, parallel is the max. The shape is a decision your code makes.
- A comprehension and a pool return the same list, so the shape is invisible in the result.
- Tuning fast workers buys nothing. Only the slowest one moves a parallel wall clock.
- Fanning out saves time, not money. The tokens are the same either way.